# Chapter 21: Quantile Neural Networks. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/21-qnn.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/21-qnn.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/21-qnn.ipynb)

In [ ]:
# The exercises of this notebook need no package besides base R

## Exercise 21.8: Quantile Regression on mtcars

Use the `mtcars` dataset to explore quantile regression. We deliberately use the raw linear model `mpg ~ disp` here, rather than the chapter's `log(mpg) ~ log(disp)` specification, to keep the fitted quantile lines interpretable directly in the original units.

a) Implement the check loss function $\rho_\tau(u) = u(\tau - I(u < 0))$ in R.

b) For the linear model `mpg ~ disp`, estimate quantile regression coefficients at $\tau \in \{0.1, 0.25, 0.5, 0.75, 0.9\}$ by minimizing the sum of check losses using `optim()`.

c) Plot the data with all five quantile regression lines overlaid. Include the OLS line for comparison.

d) At what displacement value is the gap between the 10th and 90th percentile widest? (Because the fitted quantile lines are linear in displacement, the gap $q_{0.9}(x) - q_{0.1}(x)$ is itself affine in $x$, so its maximum over the observed range necessarily occurs at one of the two endpoints, not an interior point.) What does this tell you about heteroskedasticity?

## Exercise 21.9: Heteroskedastic Quantile Estimation

Simulate data with known heteroskedasticity and verify that quantile regression captures the conditional distribution.

a) Generate $n = 500$ observations: $x_i \sim U(0, 4)$, $y_i = \sin(x_i) + \sigma(x_i) \cdot \varepsilon_i$, where $\sigma(x) = 0.1 + 0.3x$ and $\varepsilon_i \sim N(0,1)$.

b) Compute the true conditional quantiles: $Q_\tau(x) = \sin(x) + \sigma(x) \cdot \Phi^{-1}(\tau)$.

c) Fit quantile regression with a polynomial of degree 4 at $\tau \in \{0.05, 0.5, 0.95\}$. Plot the estimated quantile curves alongside the true curves.

d) What fraction of observations falls below the estimated 5th percentile curve? Is it close to 5%?

## Exercise 21.10: Choosing the Trend-Filtering Smoothing Parameter by Cross-Validation

Section 21.8 fits nonlinear quantile regression by trend filtering,
$$
\min_f \sum_{i=1}^n \rho_\tau(y_i - f(x_i)) + \lambda \sum_{i=2}^{n-1} |f_{i+1} - 2f_i + f_{i-1}|,
$$
the $k = 2$ (piecewise-linear) case, but simply fixes $\lambda = 2$. Like any penalty parameter, $\lambda$ trades bias against variance and should be tuned. This exercise selects it by cross-validation using the chapter's own check loss.

a) With `set.seed(1)`, simulate $n = 150$ points $x_i \sim U(0, 10)$ from a piecewise-linear mean with a single kink, $f_{\text{true}}(x) = 0.5x$ for $x < 5$ and $f_{\text{true}}(x) = 2.5 + 3(x - 5)$ for $x \ge 5$, with $y_i = f_{\text{true}}(x_i) + \varepsilon_i$, $\varepsilon_i \sim N(0, 0.5^2)$.

b) Building the second-difference operator as a matrix `D2()` with rows $(1, -2, 1)$ and using `optim()` with `L-BFGS-B`, fit the $k = 2$ trend filter at $\tau = 0.5$ for $\lambda \in \{0.01, 0.1, 1, 3, 10, 30, 100\}$. Plot the fitted curves for three representative $\lambda$ against the data and $f_{\text{true}}$.

c) Using 5-fold cross-validation (with `set.seed(2)` for the folds), fit on 4/5 of the points and evaluate the out-of-fold average check loss $\rho_{0.5}$ on the held-out points, predicting each held-out $x$ by linear interpolation (`approx`) of the fitted $f_i$. Report the CV score for each $\lambda$ and the selected $\hat\lambda$.

d) Repeat the CV at $\tau = 0.1$ and $\tau = 0.9$. Does the same $\lambda$ minimize the CV loss at all three levels?

e) Relate the shape of the CV-loss-versus-$\lambda$ curve to the two failure modes: interpolating the noise (small $\lambda$) versus smoothing across the kink (large $\lambda$).

## Exercise 21.11: Cosine Embeddings for the Quantile Level

Section 21.2 represents a quantile network as $F^{-1}(\tau, x) = g(\psi(x) \circ \phi(\tau))$, where $\circ$ is the Hadamard (element-wise) product and $\phi$ is the cosine embedding $\phi_j(\tau) = \mathrm{ReLU}\left(\sum_{i=0}^{n-1} \cos(\pi i \tau) w_{ij} + b_j\right)$. A single such network handles all quantile levels at once, unlike the separate per-$\tau$ fits of the previous exercises.

a) In R, implement $\phi(\tau)$ with $n = 8$ cosine frequencies and embedding dimension $m = 6$, weights $w_{ij} \sim N(0, 0.3^2)$ and biases $b_j \sim N(0, 0.3^2)$ fixed by a seed. Plot each $\phi_j(\tau)$ over $\tau \in (0, 1)$.

b) Build the implicit network $f(\tau, x) = w^\top(\psi(x) \circ \phi(\tau)) + b$, where $\psi(x)$ is a one-hidden-layer network ($1 \to m$, $\tanh$) with its own weights. Simulate the chapter's synthetic example with `set.seed(7)`: $x \sim U(-1, 1)$, $y \mid x \sim N\big(\sin(\pi x)/(\pi x),\ \exp(1 - x)/10\big)$, $n = 400$.

c) Train $f$ by minimizing $\sum_{k=1}^{9} \sum_i \rho_{\tau_k}(y_i - f(\tau_k, x_i))$ jointly over $\tau_k \in \{0.1, 0.2, \ldots, 0.9\}$ with `optim()`. Separately fit 9 explicit networks, one per $\tau_k$, each a small $\psi_k$ ($1 \to m$, $\tanh$) plus readout, with no shared $\phi$.

d) Count the free parameters of the implicit network and of the 9 explicit networks combined.

e) Evaluate both at the untrained level $\tau = 0.37$: the implicit network computes $\phi(0.37)$ directly, while the explicit approach must linearly interpolate its $\tau = 0.3$ and $\tau = 0.4$ fits. Compare both to the true quantile $f_\tau(x) = \sin(\pi x)/(\pi x) + \Phi^{-1}(\tau)\sqrt{\exp(1 - x)/10}$ on a grid of $x$. Which comparison is more meaningful: the raw prediction error, or the fact that the explicit approach has no native way to answer at $\tau = 0.37$ at all?

## Exercise 21.12: Portfolio Optimization via Quantile Neural Networks: Verifying the Kelly Criterion

Section 21.7 finds the optimal portfolio weight $\omega$ without ever specifying a closed-form expected-utility function: (1) simulate the log-return $W$ given $\omega$, (2) transform it into a utility $Z=U(W)$, (3) estimate the quantile function $F^{-1}_{Z_\omega}$, and (4) integrate over $\tau$ to obtain $\mathrm{E}\left(Z_\omega\right)$, then maximize over $\omega$. Because the chapter's example uses CARA utility and normally distributed payoffs $W$, a closed-form fractional-Kelly solution is also available, which makes the algorithm self-checking: implemented correctly, it should recover the closed-form optimum.

Use the chapter's own numbers: $r_f = 0.05$, $\mu=0.1$, $\sigma=0.25$, $\gamma=2$, so that $U(W)=-e^{-\gamma W}$ and $W(\omega) = (1-\omega)r_f+\omega R$ with $R \sim N(\mu,\sigma^2)$.

a) Set a seed. For each $\omega$ on a grid over $(0,1)$ (e.g., step $0.01$), simulate $N$ draws (e.g., $N=200{,}000$) of $W(\omega) \mid \omega \sim N\big((1-\omega)r_f+\omega\mu,\ \sigma^2\omega^2\big)$ and compute $Z^{(i)} = U(W^{(i)})$.

b) Sort the simulated $Z^{(i)}$ for each $\omega$. The sorted values approximate the quantile function $F^{-1}_{Z_\omega}(\tau)$ at $\tau \approx k/N$ for the $k$-th smallest value, standing in for the quantile neural network of step 3 of the chapter's algorithm.

c) Using the trapezoid rule, integrate the sorted values over $\tau \in (0,1)$ to estimate $\mathrm{E}\left(Z_\omega\right) = \int_0^1 F_{Z_\omega}^{-1}(\tau)\, d\tau$ for every $\omega$ on the grid.

d) Report $\hat\omega^\star = \arg\max_\omega \widehat{\mathrm{E}\left(Z_\omega\right)}$ and compare it to the chapter's closed-form $\omega^\star = (\mu-r_f)/(\sigma^2\gamma)$.

e) Reusing the same underlying standard-normal draws for every $\omega$ (common random numbers) rather than drawing a fresh sample at each grid point makes $\widehat{\mathrm{E}\left(Z_\omega\right)}$ much smoother in $\omega$. Explain why, and why this matters for the reliability of the grid search in (d).